In [1]:
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

In [3]:
load_dotenv()

if os.getenv("OPENAI_API_KEY") is None:
    raise ValueError("OPENAI_API_KEY is not set")

llm = ChatOpenAI(model='gpt-4o-mini')



## PART 1 — Basic Text Summarisation using PromptTemplate

**Task 1: Load and Prepare Text**
1. Load the document using LangChain document loaders or manual loading.
2. Print:
   - Total characters
   - Sample content preview


In [24]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [6]:
loader = PyPDFLoader('attention.pdf')
docs = loader.load()

# total characters
print(f"Total characters: {len(docs[0].page_content)}")

# sample content preview
print(f"Sample content preview: {docs[0].page_content[:100]}")

Total characters: 2857
Sample content preview: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and


**Task 2: Prompt-Based Summarization**
1. Create a PromptTemplate for summarization:
   - System instruction (role as summarizer)
   - Placeholder for input text
2. Pass the text to an LLM using LangChain.
3. Generate and print the summary.

In [7]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

In [8]:
summary_prompt = PromptTemplate(
    template="""
    Summarize the following text:
    {text}
    """,
    input_variables=["text"]
)

In [9]:
summary_chain = summary_prompt | llm
summary = summary_chain.invoke(docs[0].page_content)

In [12]:
print(f"Summary of the document: {summary.content}")

Summary of the document: Google grants permission to reproduce the tables and figures from the paper "Attention Is All You Need," authored by Ashish Vaswani and colleagues, for journalistic or scholarly works with proper attribution. The paper introduces the Transformer, a novel network architecture based solely on attention mechanisms, eliminating the need for recurrence and convolutions. Experiments in machine translation show that the Transformer outperforms existing models in quality, requiring less training time. It achieves a BLEU score of 28.4 for English-to-German translation and a state-of-the-art score of 41.8 for English-to-French. The Transformer also generalizes well to other tasks, such as English constituency parsing. The authors acknowledge equal contributions and the collaborative effort in developing the model and its implementations.


**Task 3: Prompt Variations**
Create two prompt variations:
1. Short summary (5–6 lines)
2. Bullet-point summary

Compare outputs.

**Compare**
- short prompt → tighter paragraph, good for quick skim
- bullet prompt → clearer structure / key points, easier to scan
- same model+doc, different prompt = different shape of summary (prompting matters a lot)


In [14]:
short_summary_prompt = PromptTemplate(
    template="""
    Summarize the following text in 5-6 lines:
    {text}
    """,
    input_variables=["text"]
)

bullet_point_prompt = PromptTemplate(
    template="""
    Summarize the following text in bullet points:
    {text}
    """,
    input_variables=["text"]
)

short_summary_chain = short_summary_prompt | llm
bullet_point_summary_chain = bullet_point_prompt | llm

In [15]:
short_summary = short_summary_chain.invoke(docs[0].page_content)
bullet_point_summary = bullet_point_summary_chain.invoke(docs[0].page_content)

In [16]:
print(f"Short summary: {short_summary.content}")
print("-"*100)
print(f"Bullet point summary: {bullet_point_summary.content}")

Short summary: Google grants permission to reproduce tables and figures from the paper "Attention Is All You Need" for journalistic or scholarly use with proper attribution. The paper introduces the Transformer, a novel architecture that relies entirely on attention mechanisms, eliminating the need for recurrent and convolutional neural networks. Experiments demonstrate the Transformer's superior performance on machine translation tasks, achieving state-of-the-art BLEU scores on both English-to-German and English-to-French translations, while significantly reducing training time. The model also shows promise in generalizing to other tasks, such as English constituency parsing. The authors contributed equally to the research, which was presented at the 31st Conference on Neural Information Processing Systems in 2017.
----------------------------------------------------------------------------------------------------
Bullet point summary: - Google grants permission to reproduce tables an

**Task 4: Why Stuff Chain is Needed (Conceptual)**
Answer briefly:

1. What is a stuff summarization chain? → dumps (“stuffs”) all docs/chunks into one prompt and asks the LLM to summarize once.
2. When is it suitable to use? → short docs that fit comfortably in the context window. simple + fast.
3. Limitations of stuff chain → fails / truncates / gets noisy when the doc is too long for the model context. one-shot, no intermediate compression.



**Task 5: Implement Stuff Summarization Chain**
1. Use load_summarize_chain() with:
   - chain_type="stuff"
2. Pass the entire document as input.
3. Generate summary.


In [19]:
from langchain_classic.chains.summarize import load_summarize_chain

In [21]:
stuff_chain = load_summarize_chain(llm, chain_type="stuff")
summary = stuff_chain.invoke(docs)

In [23]:
print(f"Summary of the document: {summary['output_text']}")

Summary of the document: The paper "Attention Is All You Need" introduces the Transformer, a novel neural network architecture designed for sequence transduction tasks, which relies solely on attention mechanisms without recurrent or convolutional components. This approach enhances parallelization and reduces training time significantly. The Transformer model achieves state-of-the-art results in machine translation, scoring 28.4 BLEU on the WMT 2014 English-to-German task and 41.8 on the English-to-French task, both surpassing previous models while requiring less computational resources. The model's architecture includes stacked encoder and decoder layers utilizing multi-head self-attention and position-wise feed-forward networks. Additionally, the Transformer generalizes well to tasks beyond translation, such as English constituency parsing. The authors highlight the model's efficiency and potential for future applications across various domains.


**Task 6: Comparison with Prompt-Based Summary**
Compare:
- Prompt-only summary
- Stuff chain summary

Discuss differences in output quality and limitations.

**Compare**
- prompt-only: you hand-roll `PromptTemplate | llm` and usually pass raw text / one page
- stuff chain: LangChain helper (`load_summarize_chain(..., "stuff")`) expects Document list and uses a built-in summarize prompt
- quality can be similar on small text; stuff is nicer for Document pipelines. both break on very long input for the same context-limit reason


## PART 3 — Map-Reduce Summarization Chain

**Task 7: Why Map-Reduce is Needed (Conceptual)**
Answer briefly:

1. Why large documents need map-reduce summarization? → whole PDF wont fit in one context window. map-reduce summarizes chunks first, then summarizes the summaries.
2. How map and reduce steps work → **map**: summarize each chunk independently. **reduce**: combine those partial summaries into one final summary (sometimes recursively).



**Task 8: Implement Map-Reduce Summarization Chain**
1. Split the document into chunks using text splitters.
2. Use load_summarize_chain() with:
   - chain_type="map_reduce"
3. Generate final summary.


In [25]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

split_docs = splitter.split_documents(docs)

In [26]:
map_reduce_chain = load_summarize_chain(llm, chain_type="map_reduce")
summary = map_reduce_chain.invoke(split_docs)

/Users/abhishekroy/Documents/tutedude/.venv/lib/python3.12/site-packages/langchain_openai/chat_models/base.py:564: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")


In [27]:
print(f"Summary of the document: {summary['output_text']}")

Summary of the document: The paper "Attention Is All You Need" introduces the Transformer architecture, which uses attention mechanisms for sequence transduction, eliminating the need for recurrent and convolutional networks. The Transformer achieves state-of-the-art BLEU scores in machine translation, specifically 28.4 for English-to-German and 41.8 for English-to-French, while being more efficient in training and parallelization. Its encoder-decoder structure employs stacked self-attention, fully connected layers, and incorporates techniques like multi-head attention, positional encodings, and label smoothing, which enhance performance across various NLP tasks, including constituency parsing. The research demonstrates the model's capacity to generalize well and suggests future applications in other modalities, alongside publicly available code and references to its contributions in neural machine translation and sequence modeling.


**Task 9: Analyze Map Outputs (Optional)**
1. Print intermediate summaries for each chunk.
2. Observe how the reduce step combines them.

**Note**
- map step = many small summaries (local detail)
- reduce step = merge into one global summary (less detail, more overview)


In [29]:
print(type(summary))
print(summary.keys())

<class 'dict'>
dict_keys(['input_documents', 'output_text'])


In [30]:
print(map_reduce_chain.llm_chain)
print(map_reduce_chain.reduce_documents_chain)

verbose=False prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='Write a concise summary of the following:\n\n\n"{text}"\n\n\nCONCISE SUMMARY:') llm=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-openai': '1.6.0'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions o

## PART 4 — Refine Summarization Chain

**Task 10: Understanding Refine Chain (Conceptual)**
Answer briefly:

1. How refine summarization works → summarize chunk 1, then update that summary with chunk 2, then chunk 3, … iteratively refine the running summary.
2. Difference between map-reduce and refine → map-reduce = parallel-ish map then merge. refine = sequential update (order matters, usually more coherent narrative, often slower).



**Task 11: Implement Refine Summarization Chain**
1. Use load_summarize_chain() with:
   - chain_type="refine"
2. Generate refined summary iteratively.


In [31]:
refine_chain = load_summarize_chain(llm, chain_type="refine")
summary = refine_chain.invoke(docs)
print(f"Refined summary: {summary['output_text']}")

Refined summary: The paper "Attention Is All You Need" introduces the Transformer, a groundbreaking neural network architecture that relies entirely on attention mechanisms, eliminating the need for recurrent or convolutional layers. This innovative approach significantly enhances machine translation tasks, achieving a BLEU score of 28.4 for English-to-German and 41.8 for English-to-French translations, surpassing existing models and reducing training time. In contrast to previous models that utilized recurrent networks, the Transformer establishes global dependencies between inputs and outputs through self-attention, enabling greater parallelization and efficiency in handling longer sequences.

The architecture consists of an encoder and a decoder, each composed of six identical layers. The encoder features a multi-head self-attention mechanism and a fully connected feed-forward network, complemented by residual connections and layer normalization to enhance performance. The decoder c

**Task 12: Comparison of All Summarization Methods**
Compare:
- Prompt-based
- Stuff chain
- Map-reduce chain
- Refine chain

Evaluate based on:
- Summary quality
- Coherence
- Suitability for long documents

**Rough compare**
| method | quality (small doc) | coherence | long docs |
|---|---|---|---|
| prompt-based | good if prompt is good | ok | bad (context limit) |
| stuff | similar to prompt | ok | bad (same limit) |
| map-reduce | solid overview | can feel “stitched” | good |
| refine | often smoother story | usually better flow | good but slower |

pick stuff/prompt for short text; map-reduce for long PDFs when speed matters; refine when you want iterative polish.




## PART 5 — Mini Project: Document Summarizer

**Task 13: Build a Summarization Function**
Create a reusable function:
```
def summarize_document(text, method="map_reduce"):
    # returns summary
```
Allow switching between:
- prompt
- stuff
- map_reduce
- refine


In [35]:
def summarize_document(text, method="map_reduce"):
    if method == "prompt":
        prompt = PromptTemplate(
            template="""
            Summarize the following text:
            {text}
            """,
            input_variables=["text"]
        )
        summary_chain = prompt | llm
        summary = summary_chain.invoke(text)
        return summary.content
    elif method in ["stuff", "map_reduce", "refine"]:
        chain = load_summarize_chain(llm, chain_type=method)
        summary = chain.invoke(text)
        return summary['output_text']
    else:
        raise ValueError(f"Invalid method: {method}")

In [36]:
print(summarize_document(docs[0].page_content, "prompt"))

The paper "Attention Is All You Need" by researchers from Google Brain and Google Research introduces the Transformer architecture, which relies solely on attention mechanisms, eliminating the need for recurrent or convolutional neural networks. This new approach demonstrates superior performance on machine translation tasks, achieving significant BLEU score improvements over previous models while being more efficient to train. Specifically, it reached a BLEU score of 28.4 on the WMT 2014 English-to-German task and 41.8 on the English-to-French task, with minimal training time and costs compared to existing methods. Additionally, the Transformer architecture adapts well to other tasks, such as English constituency parsing. The authors acknowledge equal contributions from various team members in developing the model and its implementations.


In [37]:
print(summarize_document(docs, "stuff"))

The paper introduces the Transformer, a novel neural network architecture based entirely on attention mechanisms, eliminating the need for recurrent or convolutional layers commonly used in sequence transduction tasks such as machine translation. The Transformer model demonstrates superior performance on the WMT 2014 English-to-German and English-to-French translation tasks, achieving state-of-the-art BLEU scores (28.4 and 41.8, respectively) while being significantly more parallelizable and requiring less training time compared to prior models. The authors highlight the benefits of self-attention, including better handling of long-range dependencies and improved computational efficiency. Additionally, the Transformer proves capable of generalizing to other tasks such as English constituency parsing. The authors conclude by expressing interest in further applications of attention-based models across different modalities.


In [38]:
print(summarize_document(docs, "map_reduce"))

/Users/abhishekroy/Documents/tutedude/.venv/lib/python3.12/site-packages/langchain_openai/chat_models/base.py:564: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")


The paper "Attention Is All You Need" introduces the Transformer, a neural network architecture that exclusively utilizes attention mechanisms, avoiding the use of recurrent and convolutional structures. This innovative model demonstrates superior performance in machine translation tasks, achieving higher BLEU scores and requiring significantly less training effort compared to traditional sequence transduction models. The Transformer features an encoder-decoder structure with stacked self-attention and fully connected layers, employing techniques like scaled dot-product attention and multi-head attention to enhance its capability to model dependencies. 

The authors detail the architecture's efficiency, noting its ability to process sequences in parallel and generalize to other tasks, such as constituency parsing. The model, trained on the WMT 2014 datasets, achieves new benchmarks in English-to-German and English-to-French translation at reduced computational costs. The research, a co

In [39]:
print(summarize_document(docs, "refine"))

The paper "Attention Is All You Need" introduces the Transformer, a revolutionary neural network architecture that relies exclusively on attention mechanisms, eliminating the need for recurrent and convolutional layers. This innovative design enhances parallelization, significantly reducing training time and addressing the inefficiencies associated with sequential computation inherent in traditional recurrent neural networks. The Transformer achieves state-of-the-art results in machine translation, setting a new BLEU score of 28.4 for English-to-German translation and 41.8 for English-to-French, significantly outperforming preceding models at a fraction of their training costs.

Central to this architecture is the concept of attention, particularly the "Scaled Dot-Product Attention" mechanism, which computes attention scores by taking the dot product of queries and keys, scaling them by the square root of their dimension, and applying a softmax function to derive weights for the corres

**Task 14: Observations & Insights**
Write short answers:

1. Best summarization strategy for very long documents → map-reduce (or refine if you can afford sequential passes). never stuff the whole thing if it blows the context window.
2. Trade-offs between speed and quality → stuff/prompt = cheapest/fastest on small text. map-reduce = more LLM calls (cost/latency) but handles length. refine = often nicest coherence, usually slowest (N sequential calls).
3. Real-world use cases of each method → prompt/stuff: emails, tickets, short blogs. map-reduce: papers, books, meeting transcripts. refine: reports where narrative consistency matters (exec summaries).
